# Hybrid Retrieval Study for Noisy Product Queries

Modern retrieval systems often struggle with noisy and unstructured user queries.
This notebook explores hybrid retrieval pipelines combining sparse retrieval (BM25)
and dense semantic retrieval methods for product matching tasks.

The study compares:
- BM25 retrieval
- Embedding-based retrieval
- Hybrid retrieval approaches

using real-world product and review data.

## Motivation

Sparse retrieval methods are efficient but often fail under terminology mismatch,
while dense retrieval methods capture semantics but may lose lexical precision.
This study explores whether hybrid retrieval improves robustness on noisy product queries.

## Pipeline Overview

1. Load and clean dataset
2. Build product corpus
3. Construct noisy query set
4. BM25 retrieval
5. Dense retrieval
6. Hybrid score fusion
7. Evaluation and failure analysis

In [1]:
import pandas as pd
import numpy as np

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\dahie\OneDrive\Masaüstü\New folder\luciel\s\7817_1.csv")

In [3]:
print("Shape:", df.shape)

print("\nColumns:\n")
print(df.columns.tolist())

Shape: (1597, 27)

Columns:

['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded', 'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer', 'manufacturerNumber', 'name', 'prices', 'reviews.date', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes', 'upc', 'weight']


In [4]:
df.head(5)

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
0,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",NaN,NaN,Cristina M,NaN,NaN,205 grams
1,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,Allow me to preface this with a little history...,One Simply Could Not Ask For More,NaN,NaN,Ricky,NaN,NaN,205 grams
2,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,4.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,NaN,NaN,Tedd Gardiner,NaN,NaN,205 grams
3,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I bought one of the first Paperwhites and have...,Love / Hate relationship,NaN,NaN,Dougal,NaN,NaN,205 grams
4,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,5.0,https://www.amazon.com/Kindle-Paperwhite-High-...,I have to say upfront - I don't like coroporat...,I LOVE IT,NaN,NaN,Miljan David Tanic,NaN,NaN,205 grams


In [5]:
df.isnull().sum().sort_values(ascending=False)

reviews.userCity        1597
reviews.userProvince    1597
sizes                   1597
reviews.doRecommend     1058
dimension               1032
weight                   911
colors                   823
ean                      699
upc                      699
reviews.numHelpful       697
manufacturerNumber       695
manufacturer             632
reviews.rating           420
reviews.date             380
reviews.username          17
reviews.title             17
reviews.sourceURLs         0
reviews.text               0
id                         0
asins                      0
name                       0
keys                       0
dateUpdated                0
dateAdded                  0
categories                 0
brand                      0
prices                     0
dtype: int64

In [6]:
df = df.dropna(subset=[
    "name",
    "reviews.text"
])
df["review_length"] = (
    df["reviews.text"]
    .astype(str)
    .str.split()
    .str.len()
)

df = df[df["review_length"] > 5]
print(df.shape)

(1572, 28)


In [7]:
df = df.drop_duplicates(
    subset=[
        "name",
        "reviews.text"
    ]
).reset_index(drop=True)

In [8]:
df.isnull().sum().sort_values(ascending=False)

reviews.userProvince    1117
reviews.userCity        1117
sizes                   1117
reviews.doRecommend      578
dimension                556
colors                   553
weight                   463
upc                      433
ean                      433
manufacturerNumber       408
manufacturer             384
reviews.numHelpful       250
reviews.rating           155
reviews.date             118
reviews.title             17
reviews.username          17
reviews.text               0
reviews.sourceURLs         0
id                         0
asins                      0
prices                     0
name                       0
keys                       0
dateUpdated                0
dateAdded                  0
categories                 0
brand                      0
review_length              0
dtype: int64

In [9]:
print(df.shape)

(1117, 28)


In [10]:
df["query"] = df["reviews.text"]

## Product-Level Corpus Construction

To avoid repeated product entries in the retrieval index,
a unique product-level corpus is constructed while preserving
review text as noisy natural-language queries.

In [11]:
product_df = (
    df[["name", "brand", "categories"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [12]:
product_df["document"] = (
    product_df["name"].fillna("") + " " +
    product_df["brand"].fillna("") + " " +
    product_df["categories"].fillna("")
)
corpus = product_df["document"].tolist()
query_df = df[["query", "name"]].copy()

In [13]:
print("Number of unique products:", len(product_df))
print("Number of queries:", len(query_df))

Number of unique products: 64
Number of queries: 1117


In [14]:
sample = query_df.sample(5)

for idx, row in sample.iterrows():

    print("=" * 80)

    print("\nQUERY:\n")
    print(row["query"])

    print("\nTRUE PRODUCT:\n")
    print(row["name"])


QUERY:

Always loved the Amazon Echo. Now we can rake it with us, outside, wherever!

TRUE PRODUCT:

Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker

QUERY:

Excellent, my daughter loves this. She can read books, do her school work, watch a few shows, and listen to music. She really enjoys having something that is just for her to use. She is 12 and this was the perfect age appropriate gift that is also extremely functional. I do use the parent settings for some extra security when she is using it unsupervised but otherwise, I do not have a set time limit. Great product.

TRUE PRODUCT:

All-New Fire 7 Kids Edition Tablet

QUERY:

I have several echo dots around my home, but this makes it easier for me to carry into a room that doesn't have one and control things around the house, listen to the news or music. It holds a great charge.

TRUE PRODUCT:

Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker

QUERY:

I am happy in having this good things.

TRUE PRODUCT:

All-New Amazon 

## Sparse Retrieval Baseline (BM25)

BM25 is a lexical retrieval algorithm based on exact token overlap.
While efficient and widely used, it may struggle under noisy user phrasing,
implicit intent, and terminology mismatch.

In [15]:
corpus = product_df["document"].tolist()
tokenized_corpus = [
    doc.lower().split()
    for doc in corpus
]

In [16]:
bm25 = BM25Okapi(tokenized_corpus)

In [17]:
test_query = query_df.iloc[0]["query"]

true_product = query_df.iloc[0]["name"]

print("TRUE PRODUCT:\n")
print(true_product)

print("\nQUERY:\n")
print(test_query)

TRUE PRODUCT:

Kindle Paperwhite

QUERY:

I initially had trouble deciding between the paperwhite and the voyage because reviews more or less said the same thing: the paperwhite is great, but if you have spending money, go for the voyage.Fortunately, I had friends who owned each, so I ended up buying the paperwhite on this basis: both models now have 300 ppi, so the 80 dollar jump turns out pricey the voyage's page press isn't always sensitive, and if you are fine with a specific setting, you don't need auto light adjustment).It's been a week and I am loving my paperwhite, no regrets! The touch screen is receptive and easy to use, and I keep the light at a specific setting regardless of the time of day. (In any case, it's not hard to change the setting either, as you'll only be changing the light level at a certain time of day, not every now and then while reading).Also glad that I went for the international shipping option with Amazon. Extra expense, but delivery was on time, with tra

In [18]:
tokenized_query = test_query.lower().split()

In [19]:
scores = bm25.get_scores(tokenized_query)

In [20]:
top_n = 5

top_indices = np.argsort(scores)[::-1][:top_n]

for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)

    print(f"Rank {rank}")
    print(f"Score: {scores[idx]:.4f}")

    print("\nDOCUMENT:\n")
    print(product_df.iloc[idx]["document"])

Rank 1
Score: 52.3181

DOCUMENT:

Kindle for Kids Bundle with the latest Kindle E-reader Amazon Amazon Devices,Kindle Accessories
Rank 2
Score: 47.5797

DOCUMENT:

Moshi Anti-Glare No Bubble Screen Protector for the Fire Phone Moshi Cell Phones & Accessories,Accessories,Screen Protectors,Cell,Amazon Devices,Electronics
Rank 3
Score: 25.3237

DOCUMENT:

Amazon 5W USB Official OEM Charger and Power Adapter for Fire Tablets and Kindle eReaders Amazon Amazon Devices & Accessories,Amazon Device Accessories,Power Adapters & Cables,Kindle Store,Kindle E-Reader Accessories,Kindle Paperwhite Accessories
Rank 4
Score: 20.2890

DOCUMENT:

Kindle Fire HDX 7" Amazon Amazon Devices,Kindle Store,buy a kindle
Rank 5
Score: 16.9402

DOCUMENT:

Alexa Voice Remote for Amazon Echo and Echo Dot Amazon Amazon Devices & Accessories,Amazon Device Accessories,Controllers & Remote Controls,Kindle Store,Amazon Echo Accessories,Remote Controls


## Initial Retrieval Observations

BM25 demonstrates strong sensitivity to lexical overlap and successfully retrieves
documents related to Kindle products and Amazon device categories.

However, the ranking quality also highlights several limitations of sparse retrieval:

- The top-ranked result is not the correct target product
- Retrieval is heavily influenced by shared surface-level tokens
- Product functionality and semantic intent are not consistently captured

For example, multiple retrieved documents contain overlapping terms such as
"Kindle", "Amazon", or "Accessories", despite representing different products.

This behavior illustrates a common limitation of lexical retrieval systems:
exact token matching does not necessarily imply semantic relevance.

In [21]:
def bm25_retrieve(query, top_k=5):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]

    return top_indices

In [22]:
correct = 0

top_k = 5

for _, row in query_df.iterrows():

    query = row["query"]
    true_product = row["name"]

    top_indices = bm25_retrieve(query, top_k=top_k)

    retrieved_products = (
        product_df.iloc[top_indices]["name"]
        .tolist()
    )

    if true_product in retrieved_products:
        correct += 1

In [23]:
recall_at_5 = correct / len(query_df)

print(f"BM25 Recall@5: {recall_at_5:.4f}")

BM25 Recall@5: 0.1674


## BM25 Evaluation Results

The BM25 baseline achieved a Recall@5 score of approximately 0.17.

This result suggests that lexical retrieval alone struggles significantly
under noisy natural-language queries and weak token overlap conditions.

Although BM25 is capable of retrieving partially related Amazon and Kindle
products, exact product matching remains difficult when user reviews rely on:

- informal phrasing
- implicit functionality descriptions
- semantic references rather than exact product terminology

These observations motivate the use of dense semantic retrieval methods,
which are designed to capture contextual and semantic similarity beyond
surface-level token matching.

# Dense Semantic Retrieval

Unlike lexical retrieval methods such as BM25,
dense retrieval systems map text into semantic vector spaces.

This allows retrieval models to capture contextual similarity
even when exact keyword overlap is weak or entirely absent.

In [24]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

#
Dense retrieval is implemented using the `all-MiniLM-L6-v2`
sentence-transformer model, which maps both product documents
and user queries into a shared semantic embedding space.
Model would perform, the warning is ignorable.

In [26]:
product_embeddings = model.encode(
    corpus,
    show_progress_bar=True
)
print(product_embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(64, 384)


Product documents are embedded into a dense semantic vector space,
allowing similarity comparisons beyond exact keyword overlap.

In [27]:
def dense_retrieve(query_embeddings, top_k=5):

    similarities = cosine_similarity(
        [query_embeddings],
        product_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return top_indices

In [28]:
queries = query_df["query"].tolist()

query_embeddings = model.encode(
    queries,
    show_progress_bar=True
)

print(query_embeddings.shape)

Batches:   0%|          | 0/35 [00:00<?, ?it/s]

(1117, 384)


In [29]:
test_query_embedding = query_embeddings[0]

top_indices = dense_retrieve(
    test_query_embedding,
    top_k=5
)

for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)

    print(f"Rank {rank}")

    print("\nDOCUMENT:\n")
    print(product_df.iloc[idx]["document"])

Rank 1

DOCUMENT:

Kindle Paperwhite Amazon Amazon Devices,mazon.co.uk
Rank 2

DOCUMENT:

Certified Refurbished Kindle Paperwhite E-reader - Black Amazon Amazon Devices
Rank 3

DOCUMENT:

Kindle Paperwhite 3G Amazon Amazon Devices
Rank 4

DOCUMENT:

Kindle Paperwhite E-reader - Black Amazon Amazon Devices
Rank 5

DOCUMENT:

Kindle Paperwhite Amazon Amazon Devices,Kindle Store,Kindle Accessories


## Dense Retrieval Observations

Dense retrieval produces substantially more semantically coherent results
than the BM25 baseline.

Instead of relying purely on token overlap, the embedding-based retrieval
pipeline groups together products that are contextually and functionally similar.

For the evaluated query, the dense retriever consistently retrieves
multiple Kindle Paperwhite variants and closely related e-reader products,
indicating that the semantic embedding space successfully captures
high-level product similarity and contextual intent.

This behavior demonstrates one of the key advantages of dense retrieval systems:
semantic relevance can be preserved even when exact lexical overlap is limited.

## Dense Retrieval Evaluation

To compare dense retrieval against the BM25 baseline,
the same Recall@5 evaluation pipeline is applied using
cosine similarity over semantic embeddings.

In [30]:
correct = 0

top_k = 5

for i, row in query_df.iterrows():

    true_product = row["name"]

    query_embedding = query_embeddings[i]

    top_indices = dense_retrieve(
        query_embedding,
        top_k=top_k
    )

    retrieved_products = (
        product_df.iloc[top_indices]["name"]
        .tolist()
    )

    if true_product in retrieved_products:
        correct += 1

In [31]:
dense_recall_at_5 = correct / len(query_df)

print(f"Dense Recall@5: {dense_recall_at_5:.4f}")

Dense Recall@5: 0.7099


## BM25 vs Dense Retrieval

The dense semantic retrieval pipeline substantially outperformed
the BM25 lexical baseline.

| Method | Recall@5 |
|---|---|
| BM25 | 0.1674 |
| Dense Retrieval | 0.7099 |

The results highlight a key limitation of sparse lexical retrieval systems:
exact token overlap alone is often insufficient under noisy natural-language queries.

Dense retrieval, on the other hand, is capable of capturing:

- semantic similarity
- contextual meaning
- implicit product references
- functional relationships between queries and products

This explains the significant improvement in retrieval quality observed
across the evaluation benchmark.

# Hybrid Retrieval

Modern retrieval systems often combine sparse lexical retrieval
with dense semantic retrieval in order to balance:

- exact keyword matching
- semantic understanding
- robustness against noisy phrasing

This section explores a simple hybrid retrieval strategy
based on weighted score fusion.

In [32]:
def bm25_scores(query):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    return np.array(scores)

In [33]:
def dense_scores(query_embedding):

    similarities = cosine_similarity(
        [query_embedding],
        product_embeddings
    )[0]

    return np.array(similarities)

In [34]:
def normalize(scores):

    scores = np.array(scores)

    return (
        scores - scores.min()
    ) / (
        scores.max() - scores.min() + 1e-8
    )

In [35]:
def hybrid_retrieve(
    query,
    query_embedding,
    top_k=5,
    alpha=0.3
):

    bm25_score_values = normalize(
        bm25_scores(query)
    )

    dense_score_values = normalize(
        dense_scores(query_embedding)
    )

    final_scores = (
        alpha * bm25_score_values
        +
        (1 - alpha) * dense_score_values
    )

    top_indices = np.argsort(
        final_scores
    )[::-1][:top_k]

    return top_indices

In [36]:
test_query = query_df.iloc[0]["query"]

test_query_embedding = query_embeddings[0]

top_indices = hybrid_retrieve(
    test_query,
    test_query_embedding,
    top_k=5
)

for rank, idx in enumerate(top_indices, start=1):

    print("=" * 80)

    print(f"Rank {rank}")

    print("\nDOCUMENT:\n")
    print(product_df.iloc[idx]["document"])

Rank 1

DOCUMENT:

Kindle Paperwhite Amazon Amazon Devices,mazon.co.uk
Rank 2

DOCUMENT:

Certified Refurbished Kindle Paperwhite E-reader - Black Amazon Amazon Devices
Rank 3

DOCUMENT:

Kindle Paperwhite 3G Amazon Amazon Devices
Rank 4

DOCUMENT:

Kindle Paperwhite E-reader - Black Amazon Amazon Devices
Rank 5

DOCUMENT:

Kindle Paperwhite Amazon Amazon Devices,Kindle Accessories


## Hybrid Retrieval Evaluation

To evaluate whether combining sparse lexical retrieval
with dense semantic retrieval improves overall retrieval quality,
the same Recall@5 benchmark is applied to the hybrid pipeline.

The hybrid retriever combines normalized BM25 scores
with dense cosine similarity scores using weighted score fusion.

In [38]:
correct = 0

top_k = 5

for i, row in query_df.iterrows():

    query = row["query"]

    true_product = row["name"]

    query_embedding = query_embeddings[i]

    top_indices = hybrid_retrieve(
        query,
        query_embedding,
        top_k=top_k
    )

    retrieved_products = (
        product_df.iloc[top_indices]["name"]
        .tolist()
    )

    if true_product in retrieved_products:
        correct += 1

In [39]:
hybrid_recall_at_5 = correct / len(query_df)

print(f"Hybrid Recall@5: {hybrid_recall_at_5:.4f}")

Hybrid Recall@5: 0.6697


# Final Retrieval Comparison

| Method | Recall@5 |
|---|---|
| BM25 | 0.1674 |
| Dense Retrieval | 0.7099 |
| Hybrid Retrieval | 0.6697 |

The dense semantic retrieval pipeline significantly outperformed
the BM25 lexical baseline across the benchmark.

While hybrid retrieval is commonly used in production retrieval systems,
the hybrid approach underperformed relative to pure dense retrieval
within this dataset.

This behavior appears to stem from the highly noisy and semantic nature
of the query distribution:

- user reviews rarely contain exact product terminology
- lexical overlap is often weak
- semantic intent dominates retrieval relevance

As a result, introducing BM25 lexical signals into the ranking pipeline
occasionally introduced retrieval noise and degraded overall ranking quality.

This outcome highlights an important retrieval engineering insight:
hybrid retrieval is not universally optimal, and retrieval effectiveness
depends heavily on the characteristics of the query-document distribution.

Future improvements may include:

- approximate nearest neighbor (ANN) search
- reranking pipelines
- cross-encoder rerankers
- query expansion
- vector databases such as FAISS or Pinecone

# Key Takeaways

- Sparse lexical retrieval struggles under noisy natural-language queries
- Dense semantic retrieval significantly improves contextual matching
- Hybrid retrieval is not universally optimal and depends heavily on dataset characteristics
- Retrieval system design requires balancing semantic understanding with lexical precision